# grf-DiD (Wang 2022) — B2_sweep (N=800)

**Workstream B2 · sample-size sweep (bias->0, var->0, sqrt(N))**

one N per notebook; combine the four N CSVs afterwards

A **CATT-capable** benchmark for R1.5 / R3.1.1, on the *same seeded panels* as
DiD-BCF, written to the DiD-BCF summary schema and scored by the engine's own
`compute_metrics` / `surface_metrics` — side by side with DiD-BCF and the other
benchmarks.

* **grf-DiD (Wang 2022)** — `method="wang"`. The "readily available"
  first-difference `grf::causal_forest` R3.1.1 names: difference the outcome
  against the last clean pre-period, then a standard causal forest per
  (cohort *g*, post period *t*) with never-treated controls. Per-obs `tau_hat(X)`
  with grf's variance estimate → a genuine CATT surface +
  `average_treatment_effect` GATT(g,t). Fast (seconds/rep) and dynamics-aware.

> **CFFE is intentionally not run** (R1.5 / R3.1.1 named both CFFE *and* a
> grf-based DiD; we satisfy the requirement with the latter). The original CFFE
> package `MACKattenberg/cffe` does **not** compile — it ships only the modified
> R + Rcpp bindings; the modified grf C++ **core** (`fe_trainer`, FE-aware `Data`,
> the FE splitting rule) was never committed and `grf/src/*` are broken text-stub
> symlinks (the repo's own build log fails on `Makevars: missing separator`). The
> Python `causalfe` library is **not a faithful** re-implementation (it omits
> grf's R-learner orthogonalisation and uses heuristic CIs), so we do not use it.
> See `README.md`.

> **Colab:** upload just this notebook and *Run all*. grf-DiD is seconds/rep;
> lower `REPS` for a quick check.

In [ ]:
# ===== Parameters (edit me) =====
SCENARIO   = "B2_sweep"
DGP        = "canonical"
N          = 800        # this notebook's sample size (200/400/800/1600)
REPS       = 100       # replications per linearity degree (matches the DiD-BCF runs)
WANG_TREES = 2000      # grf::causal_forest trees for the Wang DiD
JOBS       = 2         # parallel workers for the (cheap) data-generation step

# ===== Install R + grf (fast binary package) =====
import shutil, subprocess, os, sys
if shutil.which('Rscript') is None:
    subprocess.run('sudo apt-get -qq update && sudo apt-get -qq install -y r-base',
                   shell=True, check=True)
codename = (subprocess.run(['bash','-lc','. /etc/os-release && echo $VERSION_CODENAME'],
            capture_output=True, text=True).stdout.strip() or 'jammy')
r_install = r'''
options(repos = c(CRAN = sprintf('https://packagemanager.posit.co/cran/__linux__/%s/latest', Sys.getenv('PPM_CODENAME'))),
        HTTPUserAgent = sprintf('R/%s R (%s)', getRversion(),
            paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
pkgs <- c('grf','progress')
need <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(need)) install.packages(need)
cat('R packages ready:', paste(pkgs[pkgs %in% rownames(installed.packages())], collapse=', '), '\n')
'''
res = subprocess.run(['Rscript','-e', r_install], capture_output=True, text=True,
                     env={**os.environ, 'PPM_CODENAME': codename})
print(res.stdout[-2000:]); print(res.stderr[-800:])

In [ ]:
# ===== Clone the engine + regenerate the seeded panels =====
import os, glob, subprocess
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
if not os.path.isdir("DiD-BCF"):
    subprocess.run(["git","clone","--depth","1",REPO_URL], check=True)
ROOT = "DiD-BCF/Simulation_Studies_Revision"
%pip install -q numpy pandas joblib
subprocess.run(f"cd {ROOT} && python DGPs/data_creation_{SCENARIO}.py --all-N --reps {REPS} --jobs {JOBS}", shell=True, check=True)
folder = f"{ROOT}/R_code/{SCENARIO}_datasets/N={N}"
FOLDER = os.path.abspath(folder)
print("panels in:", FOLDER, "\n ", sorted(os.path.basename(p) for p in glob.glob(FOLDER+'/linearity_degree=*')))
# the Wang runner sorts iterations numerically, so no filename zero-padding is needed.

## Method script (written to disk, then run)

`wang_grf.R` is embedded below so the notebook is self-contained; the next cell just writes it out.

In [ ]:
%%writefile wang_grf.R
# grf-based difference-in-differences benchmark in the spirit of Wang (2022):
# first-difference the outcome against the last clean pre-period, then fit a
# standard grf::causal_forest per (cohort g, post period t) with never-treated
# controls.  This is the "readily available grf causal-forest DiD" R3.1.1 names.
#
# CATT-capable: grf returns a per-observation tau_hat(X_i) with a variance
# estimate, so we emit a *genuine* CATT surface (per-treated-obs tau_hat vs the
# true CATE) plus averaged GATT(g,t) rows (grf::average_treatment_effect) -- the
# same DiD-BCF schema the other benchmarks use, fed to
# did_bcf_revision.metrics.{compute_metrics, surface_metrics}.
#
# Reads linearity_degree=1/2/3 under the CWD and writes
#   summaries_wang_<SETTING>_lin_<d>.csv
suppressMessages({library(grf); library(progress)})
sink("output_wang.txt")
ARGS <- commandArgs(TRUE)
DGP <- if (length(ARGS) >= 1) ARGS[1] else "canonical"
SETTING <- if (length(ARGS) >= 2) ARGS[2] else "B1_baseline"
METHOD <- "wang"
N_TREES <- if (length(ARGS) >= 3) as.integer(ARGS[3]) else 2000L
REPS <- if (length(ARGS) >= 4) as.integer(ARGS[4]) else 100L

SCHEMA <- c("dgp","setting","linearity_degree","N","rep","estimand_type",
            "estimand_id","g","t","k","method","post_mean","sd","q025","q05",
            "q95","q975","p_bayes","surf_rmse","surf_mae","surf_n","surf_mape",
            "surf_cover95","surf_len95","surf_cover90","surf_len90","true")
Z95 <- 1.959964; Z90 <- 1.644854
XN <- paste0("X_", 1:5)
wald_tail <- function(est, se) if (is.finite(se) && se > 0) pnorm(-abs(est / se)) else NA_real_

new_scalar <- function(etype, eid, g, t, k, est, se, truth) {
  data.frame(estimand_type = etype, estimand_id = eid, g = g, t = t, k = k,
             post_mean = est, sd = se, q025 = est - Z95 * se, q05 = est - Z90 * se,
             q95 = est + Z90 * se, q975 = est + Z95 * se, p_bayes = wald_tail(est, se),
             surf_rmse = NA_real_, surf_mae = NA_real_, surf_n = NA_integer_,
             surf_mape = NA_real_, surf_cover95 = NA_real_, surf_len95 = NA_real_,
             surf_cover90 = NA_real_, surf_len90 = NA_real_, true = truth,
             stringsAsFactors = FALSE)
}

new_surface <- function(true_vec, est_vec, se_vec) {
  ok <- is.finite(true_vec) & is.finite(est_vec) & is.finite(se_vec)
  true_vec <- true_vec[ok]; est_vec <- est_vec[ok]; se_vec <- se_vec[ok]
  if (!length(true_vec)) return(NULL)
  err <- est_vec - true_vec
  lo95 <- est_vec - Z95 * se_vec; hi95 <- est_vec + Z95 * se_vec
  lo90 <- est_vec - Z90 * se_vec; hi90 <- est_vec + Z90 * se_vec
  nz <- abs(true_vec) > 1e-8
  data.frame(estimand_type = "CATT", estimand_id = "surface",
             g = NA_real_, t = NA_real_, k = NA_real_, post_mean = NA_real_,
             sd = NA_real_, q025 = NA_real_, q05 = NA_real_, q95 = NA_real_,
             q975 = NA_real_, p_bayes = NA_real_,
             surf_rmse = sqrt(mean(err ^ 2)), surf_mae = mean(abs(err)),
             surf_n = length(err),
             surf_mape = if (any(nz)) mean(abs(err[nz] / true_vec[nz])) else NA_real_,
             surf_cover95 = mean(lo95 <= true_vec & true_vec <= hi95),
             surf_len95 = mean(hi95 - lo95),
             surf_cover90 = mean(lo90 <= true_vec & true_vec <= hi90),
             surf_len90 = mean(hi90 - lo90), true = NA_real_,
             stringsAsFactors = FALSE)
}

finalize <- function(rows, rep, N, lin_degree) {
  if (!length(rows)) return(NULL)
  df <- do.call(rbind, rows)
  df$dgp <- DGP; df$setting <- SETTING; df$linearity_degree <- lin_degree
  df$N <- N; df$rep <- rep; df$method <- METHOD
  df[, SCHEMA]
}

# one replication -> list of schema rows
run_rep <- function(d) {
  d$first_treat_period[!is.finite(d$first_treat_period)] <- 0
  N <- length(unique(d$unit_id))
  times <- sort(unique(d$time))
  cohorts <- sort(unique(d$first_treat_period[d$first_treat_period > 0]))
  never <- d$first_treat_period == 0
  if (!any(never) || !length(cohorts)) return(NULL)
  rows <- list(); surf_true <- c(); surf_est <- c(); surf_se <- c()
  for (g in cohorts) {
    base <- if ((g - 1) %in% times) g - 1 else min(times)        # clean pre-period
    yb <- d[d$time == base, c("unit_id", "Y")]; names(yb) <- c("unit_id", "Y_base")
    post_t <- times[times >= g]
    for (tt in post_t) {
      cur <- d[d$time == tt, ]
      cur <- merge(cur, yb, by = "unit_id")
      keep <- (cur$first_treat_period == g) | (cur$first_treat_period == 0)
      cur <- cur[keep, ]
      if (sum(cur$first_treat_period == g) < 5 || sum(cur$first_treat_period == 0) < 5) next
      dY <- cur$Y - cur$Y_base
      W <- as.numeric(cur$first_treat_period == g)
      cf <- tryCatch(causal_forest(X = as.matrix(cur[, XN]), Y = dY, W = W,
                                   num.trees = N_TREES, clusters = cur$unit_id,
                                   seed = 1L),
                     error = function(e) NULL)
      if (is.null(cf)) next
      pr <- predict(cf, estimate.variance = TRUE)
      treated_idx <- which(W == 1 & cur$D == 1)
      if (!length(treated_idx)) next
      tau_i <- pr$predictions[treated_idx]
      se_i <- sqrt(pmax(pr$variance.estimates[treated_idx], 0))
      cate_i <- cur$CATE[treated_idx]
      # GATT(g,t): grf doubly-robust average over the treated cohort at t
      ate <- tryCatch(average_treatment_effect(cf, subset = (W == 1),
                                               target.sample = "treated"),
                      error = function(e) c(estimate = mean(tau_i),
                                            std.err = sqrt(mean(se_i ^ 2) / length(se_i))))
      truth <- mean(cate_i, na.rm = TRUE)
      rows[[length(rows) + 1]] <- new_scalar("GATT", sprintf("g=%g_t=%d", g, tt),
        as.numeric(g), as.integer(tt), as.integer(tt - g),
        as.numeric(ate["estimate"]), as.numeric(ate["std.err"]), truth)
      surf_true <- c(surf_true, cate_i); surf_est <- c(surf_est, tau_i)
      surf_se <- c(surf_se, se_i)
    }
  }
  s <- new_surface(surf_true, surf_est, surf_se)
  if (!is.null(s)) rows[[length(rows) + 1]] <- s
  list(rows = rows, N = N)
}

options(warn = -1)
for (lin in c("linearity_degree=1", "linearity_degree=2", "linearity_degree=3")) {
  if (!dir.exists(lin)) { cat("no", lin, "- skip\n"); next }
  lin_degree <- as.integer(sub(".*=", "", lin))
  files <- list.files(lin, pattern = "^iteration_.*csv$", full.names = TRUE)
  files <- files[order(as.integer(gsub("[^0-9]", "", basename(files))))]
  if (length(files) > REPS) files <- files[1:REPS]
  if (!length(files)) { cat("no files in", lin, "- skip\n"); next }
  all_rows <- list()
  pb <- progress_bar$new(total = length(files), format = paste0(lin, " [:bar] :current/:total"))
  for (ii in seq_along(files)) {
    pb$tick(); rep <- ii - 1
    d <- read.csv(files[ii])
    rr <- tryCatch(run_rep(d), error = function(e) NULL)
    if (is.null(rr)) next
    fr <- finalize(rr$rows, rep, rr$N, lin_degree)
    if (!is.null(fr)) all_rows[[length(all_rows) + 1]] <- fr
  }
  res <- if (length(all_rows)) do.call(rbind, all_rows) else data.frame()
  fn <- sprintf("summaries_%s_%s_lin_%d.csv", METHOD, SETTING, lin_degree)
  write.csv(res, fn, row.names = FALSE, na = "")
  cat("\nwrote", fn, "(", nrow(res), "rows )\n")
}
sink()


In [ ]:
# ===== Run grf-DiD (Wang) -> DiD-BCF-schema summaries =====
import os, subprocess
WANG = os.path.abspath("wang_grf.R")
subprocess.run(f"cd {FOLDER} && Rscript {WANG} {DGP} {SCENARIO} {WANG_TREES} {REPS}",
               shell=True, check=True)
print(open(f"{FOLDER}/output_wang.txt").read()[-2000:])

In [ ]:
# ===== Inspect + download (scored like DiD-BCF; DiD-BCF shown when present) =====
import os, glob, sys, subprocess, pandas as pd
from IPython.display import display
sys.path.insert(0, ROOT)
from did_bcf_revision.metrics import compute_metrics, surface_metrics

frames = []
for f in sorted(glob.glob(f"{FOLDER}/summaries_wang_{SCENARIO}_lin_*.csv")):
    frames.append(pd.read_csv(f)); print("loaded", os.path.basename(f))
# add the DiD-BCF posterior summaries for side-by-side, if they are in the clone
for f in sorted(glob.glob(f"{ROOT}/DiD_BCF/summaries_{SCENARIO}_lin_*.csv")):
    frames.append(pd.read_csv(f)); print("loaded DiD-BCF", os.path.basename(f))

if frames:
    summ = pd.concat(frames, ignore_index=True)
    print("\nDecomposed GATT metrics (compute_metrics):")
    display(compute_metrics(summ))
    print("\nCATT-surface metrics (surface_metrics) -- the heterogeneity headline:")
    display(surface_metrics(summ))
else:
    print("No summaries written -- check the run cell output above.")

zipname = f"Wang_{SCENARIO}_results.zip"
subprocess.run(f"cd {FOLDER} && zip -q {zipname} summaries_wang_*_lin_*.csv output_wang.txt",
               shell=True)
try:
    from google.colab import files
    files.download(f"{FOLDER}/{zipname}")
except Exception as e:
    print("(not on Colab / download skipped):", e)